In [2]:
# ================================================================
#   GURGAON REAL ESTATE PRICE PREDICTION
#   Student: v.royal2004@gmail.com  |  Category: Regression
# ================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("   GURGAON REAL ESTATE PRICE PREDICTION")
print("=" * 60)

# ================================================================
# STEP 1: LOAD DATASET
# ================================================================
np.random.seed(42)
N = 800

localities = [
    'DLF Phase 1', 'DLF Phase 2', 'DLF Phase 3', 'DLF Phase 4', 'DLF Phase 5',
    'Sector 14', 'Sector 17', 'Sector 21', 'Sector 48', 'Sector 56',
    'Sector 57', 'Golf Course Road', 'Sohna Road', 'MG Road',
    'Palam Vihar', 'South City 1', 'South City 2', 'Nirvana Country',
    'Unitech Cyber Park', 'Ardee City'
]

df = pd.DataFrame({
    'Locality'   : np.random.choice(localities, N),
    'BHK'        : np.random.choice([1, 2, 3, 4, 5], N, p=[0.05, 0.30, 0.40, 0.20, 0.05]),
    'Area_sqft'  : np.random.randint(450, 5000, N),
    'Floor'      : np.random.randint(1, 30, N),
    'Age_years'  : np.random.randint(0, 25, N),
    'Parking'    : np.random.choice([0, 1, 2], N, p=[0.15, 0.60, 0.25]),
    'Furnishing' : np.random.choice(['Furnished', 'Semi-Furnished', 'Unfurnished'], N),
    'Status'     : np.random.choice(['Ready to Move', 'Under Construction'], N, p=[0.65, 0.35]),
    'Gym'        : np.random.choice([0, 1], N, p=[0.35, 0.65]),
    'Pool'       : np.random.choice([0, 1], N, p=[0.50, 0.50]),
    'Security'   : np.random.choice([0, 1], N, p=[0.20, 0.80]),
})

# Locality premium multipliers
locality_premium = {
    'DLF Phase 1': 1.30, 'DLF Phase 2': 1.25, 'DLF Phase 3': 1.20,
    'DLF Phase 4': 1.15, 'DLF Phase 5': 1.35, 'Golf Course Road': 1.40,
    'MG Road': 1.10, 'Sector 14': 1.05, 'Sector 17': 1.00,
    'Sector 21': 0.98, 'Sector 48': 0.95, 'Sector 56': 0.97,
    'Sector 57': 0.96, 'Sohna Road': 0.92, 'Palam Vihar': 0.90,
    'South City 1': 1.08, 'South City 2': 1.06,
    'Nirvana Country': 1.12, 'Unitech Cyber Park': 1.18, 'Ardee City': 1.02
}

base_price = (
    df['Area_sqft'] * 6500
    + df['BHK'] * 200000
    + df['Parking'] * 150000
    + df['Gym'] * 80000
    + df['Pool'] * 120000
    + df['Security'] * 50000
    - df['Age_years'] * 25000
    + np.random.randint(-300000, 300000, N)
)

df['Price_Lakhs'] = (base_price / 100000) * df['Locality'].map(locality_premium)
df['Price_Lakhs'] = df['Price_Lakhs'].clip(lower=25)

print(f"\n✅ Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(df.head())

# ================================================================
# STEP 2: DATA CLEANING & PREPROCESSING
# ================================================================
print("\n--- Missing Values ---")
print(df.isnull().sum())

# Drop duplicates
df.drop_duplicates(inplace=True)

# Remove outliers using IQR on price
Q1 = df['Price_Lakhs'].quantile(0.05)
Q3 = df['Price_Lakhs'].quantile(0.95)
df = df[(df['Price_Lakhs'] >= Q1) & (df['Price_Lakhs'] <= Q3)]

print(f"\n✅ After cleaning: {df.shape[0]} rows")

# ================================================================
# STEP 3: EXPLORATORY DATA ANALYSIS (EDA)
# ================================================================
print("\n--- Descriptive Statistics ---")
print(df[['Price_Lakhs', 'Area_sqft', 'BHK', 'Age_years']].describe().round(2))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Gurgaon Real Estate — EDA', fontsize=16, fontweight='bold')

# 1. Price Distribution
axes[0, 0].hist(df['Price_Lakhs'], bins=40, color='#065A82', edgecolor='white', alpha=0.85)
axes[0, 0].set_title('Price Distribution (Lakhs)', fontsize=13)
axes[0, 0].set_xlabel('Price (₹ Lakhs)')
axes[0, 0].set_ylabel('Frequency')

# 2. Price by BHK
colors = ['#4FC3F7', '#0288D1', '#01579B', '#013E6B', '#002244']
df.boxplot(column='Price_Lakhs', by='BHK', ax=axes[0, 1],
           patch_artist=True,
           boxprops=dict(facecolor='#0288D1', color='white'),
           medianprops=dict(color='yellow', linewidth=2))
axes[0, 1].set_title('Price by BHK', fontsize=13)
axes[0, 1].set_xlabel('BHK')
axes[0, 1].set_ylabel('Price (₹ Lakhs)')
plt.sca(axes[0, 1])
plt.title('Price by BHK')

# 3. Area vs Price scatter
axes[1, 0].scatter(df['Area_sqft'], df['Price_Lakhs'], alpha=0.3,
                   color='#0288D1', edgecolors='none', s=25)
axes[1, 0].set_title('Area vs Price', fontsize=13)
axes[1, 0].set_xlabel('Area (sq ft)')
axes[1, 0].set_ylabel('Price (₹ Lakhs)')

# 4. Top 10 Localities — Avg Price
avg_loc = df.groupby('Locality')['Price_Lakhs'].mean().sort_values(ascending=False).head(10)
axes[1, 1].barh(avg_loc.index, avg_loc.values, color='#065A82')
axes[1, 1].set_title('Top 10 Localities by Avg Price', fontsize=13)
axes[1, 1].set_xlabel('Avg Price (₹ Lakhs)')
axes[1, 1].invert_yaxis()

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.close()
print("\n✅ EDA plots saved: eda_plots.png")

# ================================================================
# STEP 4: FEATURE ENGINEERING & ENCODING
# ================================================================
le_locality    = LabelEncoder()
le_furnishing  = LabelEncoder()
le_status      = LabelEncoder()

df['Locality_enc']   = le_locality.fit_transform(df['Locality'])
df['Furnishing_enc'] = le_furnishing.fit_transform(df['Furnishing'])
df['Status_enc']     = le_status.fit_transform(df['Status'])

# Price per sqft (derived feature)
df['Price_per_sqft'] = df['Price_Lakhs'] / df['Area_sqft']

FEATURES = ['Area_sqft', 'BHK', 'Floor', 'Age_years',
            'Parking', 'Gym', 'Pool', 'Security',
            'Locality_enc', 'Furnishing_enc', 'Status_enc']
TARGET = 'Price_Lakhs'

X = df[FEATURES]
y = df[TARGET]

print(f"\n✅ Features: {FEATURES}")

# ================================================================
# STEP 5: TRAIN / TEST SPLIT & SCALING
# ================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"\n✅ Train: {X_train.shape[0]} samples  |  Test: {X_test.shape[0]} samples")

# ================================================================
# STEP 6: MODEL TRAINING & EVALUATION
# ================================================================
def train_evaluate(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    r2   = r2_score(y_te, preds)
    mae  = mean_absolute_error(y_te, preds)
    rmse = np.sqrt(mean_squared_error(y_te, preds))
    cv   = cross_val_score(model, X_tr, y_tr, cv=5, scoring='r2').mean()
    print(f"\n  {'─'*38}")
    print(f"  Model     : {name}")
    print(f"  R² Score  : {r2:.4f}")
    print(f"  CV R²     : {cv:.4f}")
    print(f"  MAE       : ₹ {mae:.2f} Lakhs")
    print(f"  RMSE      : ₹ {rmse:.2f} Lakhs")
    return model, preds, r2, mae, rmse

print("\n" + "=" * 42)
print("   MODEL TRAINING & EVALUATION")
print("=" * 42)

MODELS = {
    "Linear Regression"    : LinearRegression(),
    "Ridge Regression"     : Ridge(alpha=1.0),
    "Lasso Regression"     : Lasso(alpha=0.5),
    "Random Forest"        : RandomForestRegressor(n_estimators=150, random_state=42, n_jobs=-1),
    "Gradient Boosting"    : GradientBoostingRegressor(n_estimators=150, learning_rate=0.1, random_state=42),
}

results = {}
best_model, best_preds, best_r2, best_name = None, None, 0, ""

for name, model in MODELS.items():
    m, preds, r2, mae, rmse = train_evaluate(name, model,
                                              X_train_sc, X_test_sc,
                                              y_train, y_test)
    results[name] = {'r2': r2, 'mae': mae, 'rmse': rmse, 'model': m, 'preds': preds}
    if r2 > best_r2:
        best_r2, best_model, best_preds, best_name = r2, m, preds, name

print(f"\n\n🏆 Best Model: {best_name}  (R² = {best_r2:.4f})")

# ================================================================
# STEP 7: VISUALIZATIONS — MODEL COMPARISON + ACTUAL vs PREDICTED
# ================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Model Comparison
names  = list(results.keys())
r2s    = [results[n]['r2'] for n in names]
colors_bar = ['#B0BEC5' if n != best_name else '#065A82' for n in names]
axes[0].barh(names, r2s, color=colors_bar, edgecolor='white')
axes[0].set_xlim(0, 1)
axes[0].set_title('Model Comparison — R² Score', fontsize=13, fontweight='bold')
axes[0].set_xlabel('R² Score')
for i, v in enumerate(r2s):
    axes[0].text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=10)

# Actual vs Predicted
axes[1].scatter(y_test, best_preds, alpha=0.4, color='#0288D1', s=25, edgecolors='none')
mn = min(y_test.min(), best_preds.min())
mx = max(y_test.max(), best_preds.max())
axes[1].plot([mn, mx], [mn, mx], 'r--', lw=2, label='Perfect Prediction')
axes[1].set_title(f'Actual vs Predicted — {best_name}', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Actual Price (₹ Lakhs)')
axes[1].set_ylabel('Predicted Price (₹ Lakhs)')
axes[1].legend()

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.close()
print("\n✅ Model results plot saved: model_results.png")

# ================================================================
# STEP 8: FEATURE IMPORTANCE
# ================================================================
rf_model = results['Random Forest']['model']
importance_df = pd.DataFrame({
    'Feature'   : FEATURES,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(9, 5))
plt.barh(importance_df['Feature'], importance_df['Importance'],
         color='#065A82', edgecolor='white')
plt.title('Feature Importance — Random Forest', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Feature importance plot saved: feature_importance.png")

# ================================================================
# STEP 9: PREDICT A NEW PROPERTY
# ================================================================
print("\n" + "=" * 50)
print("   PREDICT A NEW PROPERTY IN GURGAON")
print("=" * 50)

new_property = {
    'Area_sqft'    : 1800,
    'BHK'          : 3,
    'Floor'        : 7,
    'Age_years'    : 3,
    'Parking'      : 1,
    'Gym'          : 1,
    'Pool'         : 1,
    'Security'     : 1,
    'Locality_enc' : le_locality.transform(['Golf Course Road'])[0],
    'Furnishing_enc': le_furnishing.transform(['Semi-Furnished'])[0],
    'Status_enc'   : le_status.transform(['Ready to Move'])[0],
}

sample_df = pd.DataFrame([new_property])[FEATURES]
sample_sc = scaler.transform(sample_df)
predicted  = best_model.predict(sample_sc)[0]

print(f"\n  Property Details:")
print(f"  Locality    : Golf Course Road")
print(f"  BHK         : 3")
print(f"  Area        : 1800 sq ft")
print(f"  Floor       : 7th")
print(f"  Age         : 3 years")
print(f"  Furnishing  : Semi-Furnished")
print(f"  Amenities   : Gym, Pool, Parking, Security")
print(f"\n  💰 Predicted Price : ₹ {predicted:.2f} Lakhs")
print(f"  💰 Approx          : ₹ {predicted/100:.2f} Crores")

print("\n" + "=" * 60)
print("   PROJECT COMPLETE ✅")
print("=" * 60)


   GURGAON REAL ESTATE PRICE PREDICTION

✅ Dataset loaded: 800 rows, 12 columns
      Locality  BHK  Area_sqft  Floor  Age_years  Parking      Furnishing  \
0    Sector 17    3       1450      9          4        0     Unfurnished   
1   Ardee City    4       1300     27         12        1     Unfurnished   
2  Palam Vihar    1       1838     28          3        1     Unfurnished   
3    Sector 57    5       4162     23          1        1  Semi-Furnished   
4    Sector 21    3       1817     25         15        0       Furnished   

               Status  Gym  Pool  Security  Price_Lakhs  
0       Ready to Move    1     0         1   101.682700  
1       Ready to Move    1     0         1    93.934952  
2  Under Construction    1     0         1   108.597951  
3       Ready to Move    1     1         1   273.680208  
4       Ready to Move    0     0         1   121.072199  

--- Missing Values ---
Locality       0
BHK            0
Area_sqft      0
Floor          0
Age_years      0
